<a href="https://colab.research.google.com/github/ergdipesh/BootCampAssignmentDP/blob/main/non_instruction_finetuning_dp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Clean up existing installations to prevent conflicts ---
print("Uninstalling existing torch, torchvision, torchaudio, and related packages...")
!pip uninstall -y torch torchvision torchaudio
!pip uninstall -y trl transformers datasets accelerate peft bitsandbytes pypdf

# --- Install Unsloth and its core dependencies ---
# Unsloth is designed to manage its own core dependencies (torch, torchvision, transformers, trl, datasets).
# Installing it first ensures these are set to compatible versions for Unsloth.
print("Installing unsloth and its core dependencies...")
!pip install -U unsloth

# --- Install other required libraries that are not directly managed by unsloth's core dependency tree ---
# These packages are generally less version-sensitive to unsloth's specific torch/transformers versions.
# Note: `trl`, `transformers`, and `datasets` are intentionally removed from this line
#       as `unsloth` should handle their installation to compatible versions.
print("Installing remaining required libraries (accelerate, peft, bitsandbytes, pypdf)...")
!pip install accelerate peft bitsandbytes pypdf

# --- IMPORTANT: torchaudio compatibility ---
# Unsloth installs `torch==2.10.0`. As noted previously, `torchaudio==2.10.0` for `cu121` is not available.
# Attempting to install other `torchaudio` versions (e.g., `2.5.1`) may cause downgrades of `torch` or
# lead to further incompatibilities. If `torchaudio` is strictly needed, further investigation
# into a compatible `torch`/`torchaudio`/`unsloth` setup for your exact use case might be required.
# For now, `torchaudio` is omitted from the installation to avoid introducing new conflicts
# with `unsloth`'s core stack. If you need `torchaudio`, you might need to adjust the `unsloth` version
# or use a different `torch` environment that offers `torch==2.x.x` and `torchaudio==2.x.x` with matching CUDA versions.

print("Dependency installation complete. IMPORTANT: Please restart your Colab runtime (Runtime -> Restart runtime) now.")

Uninstalling existing torch, torchvision, torchaudio, and related packages...
Installing unsloth and its core dependencies...
  Using cached torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metada

In [1]:
# Verify versions after installation:
import sys
print("Python:", sys.version)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
#Load raw domain text from PDF
from pathlib import Path
from pypdf import PdfReader

PDF_PATH = Path("data/non_instruction_data.pdf")
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"{PDF_PATH} was not found. Keep this notebook in the project root "
        "with the data folder beside it."
    )

reader = PdfReader(str(PDF_PATH))
raw_text = "\n".join((page.extract_text() or "") for page in reader.pages)

print("Pages:", len(reader.pages))
print("Characters extracted:", len(raw_text))
print(raw_text[:1200])

Pages: 6
Characters extracted: 15985
HR Policy Assistant: Non-Instruction Domain Corpus
Educational synthetic corpus containing 61 policy-style paragraphs. It is inspired by common public employee-handbook structures
but does not represent any real employer's policy.
1. Purpose and scope. This sample HR policy corpus is designed for a fictional organization named
Northstar Services. It is intended solely for machine-learning education and demonstration. It reflects
common employee-handbook structures and should not be treated as legal advice or as the policy of any
real employer.
2. Policy administration. Human Resources maintains the official policy library, publishes approved
revisions, and answers questions about interpretation. When a local law, collective agreement,
employment contract, or benefit-plan document provides a greater entitlement, the applicable controlling
document takes priority over this sample policy.
3. Employee responsibility. Employees are expected to read curre

In [3]:
#Clean and chunk the text
import re
from datasets import Dataset

def clean_text(text: str) -> str:
    # Normalize whitespace while preserving paragraph boundaries.
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Remove page-number-only lines when present.
    text = re.sub(r"(?m)^\s*Page\s+\d+\s*$", "", text)
    return text.strip()

cleaned_text = clean_text(raw_text)

def chunk_by_tokens(text: str, tokenizer, max_tokens: int = 512, overlap: int = 64):
    if overlap >= max_tokens:
        raise ValueError("overlap must be smaller than max_tokens")
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    step = max_tokens - overlap
    for start in range(0, len(token_ids), step):
        ids = token_ids[start:start + max_tokens]
        if len(ids) < 32:
            continue
        chunks.append(tokenizer.decode(ids, skip_special_tokens=True))
        if start + max_tokens >= len(token_ids):
            break
    return chunks

print("Cleaned characters:", len(cleaned_text))
print(cleaned_text[:800])


Cleaned characters: 15984
HR Policy Assistant: Non-Instruction Domain Corpus
Educational synthetic corpus containing 61 policy-style paragraphs. It is inspired by common public employee-handbook structures
but does not represent any real employer's policy.
1. Purpose and scope. This sample HR policy corpus is designed for a fictional organization named
Northstar Services. It is intended solely for machine-learning education and demonstration. It reflects
common employee-handbook structures and should not be treated as legal advice or as the policy of any
real employer.
2. Policy administration. Human Resources maintains the official policy library, publishes approved
revisions, and answers questions about interpretation. When a local law, collective agreement,
employment contract, or benefit-plan document provides 


In [4]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-0.5B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 1024

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for this Unsloth training notebook.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # Unsloth selects fp16 or bf16 based on the GPU.
)

print("Loaded:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.13.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded: unsloth/Qwen2.5-0.5B-unsloth-bnb-4bit
GPU: Tesla T4


In [6]:
#Apply LoRA/QLoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

Unsloth 2026.7.2 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [7]:
#Create the language-modeling dataset
chunks = chunk_by_tokens(
    cleaned_text,
    tokenizer=tokenizer,
    max_tokens=MAX_SEQ_LENGTH - 32,
    overlap=96,
)

dataset = Dataset.from_dict({"text": chunks})
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print("Example training chunk:\n")
print(dataset["train"][0]["text"][:1200])

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 3
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1
    })
})
Example training chunk:


proper accounting or violates tax rules.
37. Work-from-home eligibility. Remote work is available only for roles and employees that meet
operational, performance, security, and location requirements. Approval may be modified or withdrawn
when business needs change.
38. Remote-work request. Employees should submit a work-from-home request through the approved
process and identify the requested schedule, work location, contact details, and any equipment needs. A
request is not approved until the designated approver confirms it.
39. Remote-work availability. Employees working from home must remain reachable during agreed
working hours, attend required meetings, meet performance expectations, and update their status when
temporarily unavailable.
40. Remote-work security. Employees must protect confi

In [8]:
#test the base model before training
from transformers import TextStreamer

def generate_completion(prompt: str, max_new_tokens: int = 100) -> str:
    FastLanguageModel.for_inference(model)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

test_prompts = [
    "HR policy: Sick leave must be applied",
    "Attendance correction procedure:",
    "A reimbursement claim should include",
]

for prompt in test_prompts:
    print("=" * 80)
    print(generate_completion(prompt))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

HR policy: Sick leave must be applied for in advance

The sick leave policy is a key part of the HR system. It is important to ensure that employees are aware of their rights and responsibilities when it comes to sick leave.

What is sick leave?

Sick leave is an entitlement that allows employees to take time off work due to illness or injury. It is a form of paid leave that can be used to cover medical expenses, such as prescriptions or treatments.

How does sick leave work?

Employees who are eligible for sick leave must apply


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Attendance correction procedure: A. Correct the patient's posture and position B. Correct the patient's body position C. Correct the patient's head position D. Correct the patient's limbs position E. Correct the patient's body position
Answer:
A

The main components of a transformer are ____
A. Iron core, windings, insulation
B. Iron core, windings, oil tank
C. Windings, oil tank, insulating paper
D. Iron core, windings, oil tank, ins
A reimbursement claim should include the following information: 
A. The name of the hospital or medical institution
B. The name and contact details of the claimant
C. The specific reason for the claim
D. The amount of the claim
E. All of the above
Answer:
E

Which of the following statements about the relationship between the number of people in a group and the number of people in each subgroup is correct?
A. The larger the number of people in a group, the fewer the number of


In [11]:
#train on raw domain text
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    output_dir="outputs/non_instruction_checkpoints",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=2,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
)

train_result = trainer.train()
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

num_proc must be <= 3. Reducing num_proc to 3 for dataset of size 3.


Unsloth: Packing train dataset (num_proc=3):   0%|          | 0/3 [00:00<?, ? examples/s]

num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
3,2.673661,3.327372


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in outputs/non_instruction_checkpoints/checkpoint-3/tokenizer_config.json.


TrainOutput(global_step=3, training_loss=2.750722964604696, metrics={'train_runtime': 11.4189, 'train_samples_per_second': 0.788, 'train_steps_per_second': 0.263, 'total_flos': 19643188469760.0, 'train_loss': 2.750722964604696, 'epoch': 3.0})

In [12]:
#save the LoRA adapter and tokenizer
ADAPTER_DIR = "outputs/hr_policy_stage1_lora"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Saved adapter and tokenizer to:", ADAPTER_DIR)

Unsloth: Restored added_tokens_decoder metadata in outputs/hr_policy_stage1_lora/tokenizer_config.json.


Saved adapter and tokenizer to: outputs/hr_policy_stage1_lora


In [14]:
#test after non-instruction fine-tuning
for prompt in test_prompts:
    print("=" * 80)
    print(generate_completion(prompt))

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


HR policy: Sick leave must be applied for in advance

The sick leave policy is a key part of the HR system. It is important to ensure that employees are aware of their entitlements and how they can apply for sick leave.

What is sick leave?

Sick leave is an entitlement that allows employees to take time off work due to illness or injury. It is a form of paid leave that is available to all employees, regardless of their position or role.

How does sick leave work?

Employees who are eligible for sick leave must apply


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Attendance correction procedure: A. Correct the patient's posture, position, and body position B. Correct the patient's position, position, and body position C. Correct the patient's body position, position, and body position D. Correct the patient's body position, position, and body position E. Correct the patient's body position, position, and body position
Answer:
A

The main components of a transformer are ____
A. Iron core, windings, insulation
B. Iron core, windings,
A reimbursement claim should include the following items: 
A. The name of the claimant
B. The name of the claimant's unit
C. The name of the claimant's department
D. The name of the claimant's position
E. The name of the claimant's affiliated unit
Answer:
ABCDE

Which of the following statements about the relationship between the state and the market are correct?
A. The state is the foundation of the market.
B. The market is the foundation of the


### `requirements.txt` for this notebook

In [ ]:
%%writefile requirements.txt
unsloth
accelerate
peft
bitsandbytes
pypdf


This file contains the primary dependencies that you explicitly install. When `unsloth` is installed, it also brings along compatible versions of `transformers`, `trl`, and `datasets`.